In [1]:
import pandas as pd
import numpy as np

In [2]:
def run_cleaning_and_merge():
    print("Starting Data Cleaning and Merging...")
    import pandas as pd
    
    # 1. Load data
    employees = pd.read_csv('../data/employees.csv')
    onboarding = pd.read_csv('../data/onboarding.csv')
    support_tickets = pd.read_csv('../data/support_tickets.csv')
    tool_usage = pd.read_csv('../data/tool_usage.csv')
    
    # 2. Clean Data
    median_res_hours = support_tickets['resolution_hours'].median()
    support_tickets['resolution_hours'] = support_tickets['resolution_hours'].fillna(median_res_hours)
    
    # 3. Aggregate Support Tickets per Employee
    ticket_agg = support_tickets.groupby('employee_id').agg(
        total_tickets=('ticket_id', 'count'),
        high_priority_tickets=('priority', lambda x: (x.isin(['High', 'Critical'])).sum()),
        avg_resolution_hours=('resolution_hours', 'mean')
    ).reset_index()
    
    # 4. Aggregate Tool Usage per Employee (Week 1 & 2 ONLY to prevent leakage)
    tool_usage['date'] = pd.to_datetime(tool_usage['date'])
    first_date = tool_usage.groupby('employee_id')['date'].transform('min')
    tool_usage['days_since_start'] = (tool_usage['date'] - first_date).dt.days
    
    early_tools = tool_usage[tool_usage['days_since_start'] <= 14].copy()
    
    # NEW FEATURE ENGINEERING: Extract missing signals mentioned in README
    early_tools['login_count'] = pd.to_numeric(early_tools['login_count'], errors='coerce').fillna(0)
    early_tools['active_minutes'] = pd.to_numeric(early_tools['active_minutes'], errors='coerce').fillna(0)
    
    tool_agg = early_tools.groupby('employee_id').agg(
        early_tool_actions=('usage_id', 'count'),
        early_unique_tools=('feature_used', 'nunique'),
        early_login_count=('login_count', 'sum'),
        early_active_minutes=('active_minutes', 'sum')
    ).reset_index()
    
    # 5. Merge everything together
    master_df = employees.merge(onboarding, on='employee_id', how='left')
    master_df = master_df.merge(ticket_agg, on='employee_id', how='left')
    master_df = master_df.merge(tool_agg, on='employee_id', how='left')
    
    # Fill missing values
    master_df['total_tickets'] = master_df['total_tickets'].fillna(0)
    master_df['high_priority_tickets'] = master_df['high_priority_tickets'].fillna(0)
    master_df['early_tool_actions'] = master_df['early_tool_actions'].fillna(0)
    master_df['early_unique_tools'] = master_df['early_unique_tools'].fillna(0)
    master_df['early_login_count'] = master_df['early_login_count'].fillna(0)
    master_df['early_active_minutes'] = master_df['early_active_minutes'].fillna(0)
    
    print(f"Master Dataset Shape: {master_df.shape}")
    master_df.to_csv('master_data.csv', index=False)
    print("\nSaved 'master_data.csv'.")


In [3]:
if __name__ == '__main__':
    run_cleaning_and_merge()

Starting Data Cleaning and Merging...


Master Dataset Shape: (1470, 25)

Saved 'master_data.csv'.
